# 01 — Data Exploration

Plot processed GeoTIFF tiles alongside their ground-truth YOLO OBB labels.
All logic is imported from ; this notebook contains no business logic.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to path so  is importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

# ── Third-party ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import rasterio
from matplotlib.collections import PatchCollection

# ── Project ───────────────────────────────────────────────────────────────
from src.utils.config import load_config
from src.evaluation.metrics import parse_label_file, count_split_labels

plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("Imports OK.")

## 1. Configuration

Change the paths below to match your environment.

In [ ]:
cfg          = load_config("../configs/data_prep.yaml")
PROCESSED    = Path("../data/processed")
IMAGES_DIR   = PROCESSED / "images" / "train"
LABELS_DIR   = PROCESSED / "labels" / "train"
TILE_SIZE    = cfg.tiling.tile_size   # pixels (pre-upsample)

# Class names from the dataset yaml
import yaml
with open("../data/dataset.yaml") as f:
    _ds = yaml.safe_load(f)
CLASS_NAMES = _ds["names"]   # dict  {0: "Pirogue", 1: ..., }

print(f"Images dir : {IMAGES_DIR}")
print(f"Labels dir : {LABELS_DIR}")
print(f"Tile size  : {TILE_SIZE} px")
print(f"Classes    : {CLASS_NAMES}")

## 2. Dataset Statistics

Per-class annotation counts across all splits.

In [ ]:
from src.evaluation.metrics import build_distribution_dataframe

df_dist = build_distribution_dataframe(
    labels_root=PROCESSED / "labels",
    splits=["train", "val", "test"],
    class_names=CLASS_NAMES,
)
print(df_dist.to_string(index=False))

In [ ]:
# Bar chart
splits   = ["train", "val", "test"]
classes  = list(CLASS_NAMES.values())
x        = np.arange(len(classes))
width    = 0.25

fig, ax = plt.subplots(figsize=(12, 4))
for i, split in enumerate(splits):
    vals = df_dist[split.capitalize()].values if split.capitalize() in df_dist else [0]*len(classes)
    ax.bar(x + i * width, vals, width, label=split.capitalize())

ax.set_xticks(x + width)
ax.set_xticklabels(classes, rotation=30, ha="right")
ax.set_ylabel("Object count")
ax.set_title("Class Distribution per Split", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Tile Viewer

Display a random sample of annotated tiles with OBB overlays.
Each colour corresponds to one class.

In [ ]:
import random

CLASS_COLORS = {
    0: "#e41a1c", 1: "#377eb8", 2: "#4daf4a",
    3: "#ff7f00", 4: "#984ea3", 5: "#a65628",
}

def read_geotiff_as_rgb(path: Path) -> np.ndarray:
    """Read a GeoTIFF tile and return a (H, W, 3) uint8 array."""
    with rasterio.open(path) as ds:
        data = ds.read()          # (C, H, W)
    # Always return exactly 3 channels for display.
    if data.shape[0] >= 3:
        img = data[:3].transpose(1, 2, 0)
    else:
        img = np.repeat(data[:1].transpose(1, 2, 0), 3, axis=2)
    return img.astype(np.uint8)


def plot_tiles(tile_paths, labels_dir, class_names, tile_size, n_cols=4):
    n = len(tile_paths)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.array(axes).flatten()

    for ax, tp in zip(axes, tile_paths):
        img = read_geotiff_as_rgb(tp)
        h, w = img.shape[:2]
        ax.imshow(img)

        lbl_path = labels_dir / (tp.stem + ".txt")
        boxes = parse_label_file(lbl_path, w, h)
        for cid, pts in boxes:
            color = CLASS_COLORS.get(cid, "yellow")
            poly  = plt.Polygon(pts, fill=False, edgecolor=color, linewidth=1.5)
            ax.add_patch(poly)

        ax.set_title(tp.stem[-20:], fontsize=7)
        ax.axis("off")

    legend_handles = [
        mpatches.Patch(color=CLASS_COLORS.get(cid, "yellow"), label=name)
        for cid, name in class_names.items()
    ]
    fig.legend(handles=legend_handles, loc="lower center",
               ncol=len(class_names), fontsize=8, framealpha=0.7)
    plt.tight_layout()
    plt.show()


# Sample N annotated tiles
all_tiles  = sorted(IMAGES_DIR.glob("*.tif"))
anno_tiles = [
    t for t in all_tiles
    if (LABELS_DIR / (t.stem + ".txt")).exists()
    and (LABELS_DIR / (t.stem + ".txt")).stat().st_size > 0
]
sample = random.sample(anno_tiles, min(8, len(anno_tiles)))
plot_tiles(sample, LABELS_DIR, CLASS_NAMES, TILE_SIZE)

## 4. GeoTIFF Spatial Metadata Inspection

Confirm that each tile carries its own CRS and Affine transform.

In [ ]:
sample_tile = anno_tiles[0] if anno_tiles else all_tiles[0]

with rasterio.open(sample_tile) as ds:
    print(f"File      : {sample_tile.name}")
    print(f"CRS       : {ds.crs}")
    print(f"Transform : {ds.transform}")
    print(f"Size      : {ds.width} x {ds.height} px")
    print(f"TIFF tags : {ds.tags()}")